# Hyperparameter variants (Table A2)

Re-creates Table 1 with five CCF-Net variants added, each changing one setting:

| Label | Change from CCF-Net (`CCF_Net_u110_l4_d0.0_w40`) |
|---|---|
| Variant 1 | dropout 0.1 |
| Variant 2 | unlimited training years |
| Variant 3 | 35 training years |
| Variant 4 | 256 nodes per layer |
| Variant 5 | 5 hidden layers |

The variants were trained on 6 jump-off years (1985, 1990, ..., 2010), so the
table is scored on that subset of the cohorts in Table 1. As in Table 1, KS
p-values come from a country-level paired permutation test.

## Load forecasts

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp

DATA_DIR = '../../data'
SENS_DIR = f'{DATA_DIR}/sensitivity_checks'

# label -> forecast file, observed file, directory
MODEL_REGISTRY = {
    'CCF-Net':            dict(pred='dl_forecasts_CCF_Net_u110_l4_d0.0_w40',  obs='dl_obs_cohort_CCF_Net_u110_l4_d0.0_w40', dir=DATA_DIR),
    'Variant 1':          dict(pred='dl_forecasts_u110_l4_d0.1_w40',    obs='dl_obs_cohort_u110_l4_d0.1_w40',   dir=SENS_DIR),
    'Variant 2':          dict(pred='dl_forecasts_u110_l4_d0.0_wNone',  obs='dl_obs_cohort_u110_l4_d0.0_wNone', dir=SENS_DIR),
    'Variant 3':          dict(pred='dl_forecasts_u110_l4_d0.0_w35',    obs='dl_obs_cohort_u110_l4_d0.0_w35',   dir=SENS_DIR),
    'Variant 4':          dict(pred='dl_forecasts_u256_l4_d0.0_w40',    obs='dl_obs_cohort_u256_l4_d0.0_w40',   dir=SENS_DIR),
    'Variant 5':          dict(pred='dl_forecasts_u110_l5_d0.0_w40',    obs='dl_obs_cohort_u110_l5_d0.0_w40',   dir=SENS_DIR),
    'Freeze rates':       dict(pred='freeze_forecasts_cohort',          obs='freeze_obs_cohort',                dir=DATA_DIR),
    'Myrskyla et al.':    dict(pred='myrskyla_forecasts_cohort',        obs='myrskyla_obs_cohort',              dir=DATA_DIR),
    'de Beer':            dict(pred='debeer2_forecasts_cohort',         obs='debeer2_obs_cohort',               dir=DATA_DIR),
    'Schmertmann et al.': dict(pred='schmertmann2013_forecasts_cohort', obs='schmertmann2013_obs_cohort',       dir=DATA_DIR),
    'Sevcikova et al.':   dict(pred='sevcikova_forecasts_cohort',       obs='sevcikova_obs_cohort',             dir=DATA_DIR),
    'Lee':                dict(pred='lee2_forecasts_cohort',            obs='lee2_obs_cohort',                  dir=DATA_DIR),
}

# Completed fertility = sum of ASFR over ages 15-40 (CF40), as in Bohk-Ewald,
# Li & Myrskyla (2018). Only these ages are scored.
CF_AGE_LO, CF_AGE_HI = 15, 40
N_CF_AGES = CF_AGE_HI - CF_AGE_LO + 1

KEY_COLS = ['Country', 'Year', 'Age', 'JumpOffYear']   # Year is the birth cohort
OBS_KEY = ['Country', 'Year', 'Age']   # observed rates don't depend on the JOY

M = {}
for label, meta in MODEL_REGISTRY.items():
    pred = pd.read_csv(f"{meta['dir']}/{meta['pred']}.csv")
    obs = (pd.read_csv(f"{meta['dir']}/{meta['obs']}.csv")
           .drop_duplicates(OBS_KEY).drop(columns='JumpOffYear', errors='ignore'))
    # keep forecast cells only (period = Year + Age after the JOY), scored ages only
    pred = pred[(pred['Year'] + pred['Age'] > pred['JumpOffYear']) &
                pred['Age'].between(CF_AGE_LO, CF_AGE_HI)]
    obs = obs[obs['Age'].between(CF_AGE_LO, CF_AGE_HI)]
    M[label] = {'pred': pred, 'obs': obs}

# Keep only forecast cells that every method produces.
common_keys = None
for m in M.values():
    k = m['pred'][KEY_COLS].drop_duplicates()
    common_keys = k if common_keys is None else common_keys.merge(k, on=KEY_COLS)
for m in M.values():
    m['pred'] = m['pred'].merge(common_keys, on=KEY_COLS)
print(f'Common grid: {len(common_keys):,} forecast cells across {len(M)} methods')

## Completed cohort fertility errors

In [ ]:
# Absolute percentage error (APE) of completed cohort fertility, following
# Bohk-Ewald, Li & Myrskyla (2018):
#   AAF          = the cohort's age in the jump-off year (truncation age);
#   forecast CF  = observed rates for ages 15..AAF + forecast rates for AAF+1..40;
#   APE          = |forecast CF - observed CF| / observed CF * 100,
# one error per (country, birth cohort, JOY), for truncation ages 20-39 and only
# for cohorts that are complete in the data.
AAF_LO, AAF_HI = 20, 39


def cf_ape(obs_df, pred_df):
    obs = obs_df[['Country', 'Year', 'Age', 'Rate']]
    n_obs_ages = obs.groupby(['Country', 'Year'])['Age'].transform('nunique')
    true_cf = obs[n_obs_ages == N_CF_AGES].groupby(['Country', 'Year'])['Rate'].sum()

    apes = {}
    for joy in sorted(pred_df['JumpOffYear'].unique()):
        obs_hist = obs[obs['Year'] + obs['Age'] <= joy]
        fcast = pred_df[pred_df['JumpOffYear'] == joy][['Country', 'Year', 'Age', 'Rate']]
        traj = pd.concat([obs_hist, fcast])
        n_ages = traj.groupby(['Country', 'Year'])['Age'].transform('nunique')
        fcast_cf = traj[n_ages == N_CF_AGES].groupby(['Country', 'Year'])['Rate'].sum()

        for (country, byear) in fcast_cf.index.intersection(true_cf.index):
            if AAF_LO <= joy - byear <= AAF_HI:
                obs_v, fc_v = true_cf[(country, byear)], fcast_cf[(country, byear)]
                apes[(country, byear, joy)] = abs(fc_v - obs_v) / obs_v * 100
    return pd.Series(apes, name='APE')


ape_raw = {label: cf_ape(m['obs'], m['pred']) for label, m in M.items()}

# Equal testing dataset: only the CF forecasts that every method has.
common_cohorts = None
for s in ape_raw.values():
    common_cohorts = s.index if common_cohorts is None else common_cohorts.intersection(s.index)
ape_by_model = {label: s.loc[common_cohorts] for label, s in ape_raw.items()}

_idx = pd.DataFrame(list(common_cohorts), columns=['Country', 'Cohort', 'JumpOffYear'])
print(f'Equal testing dataset: {len(common_cohorts):,} CF forecasts '
      f'({_idx.Country.nunique()} countries, jump-off years '
      f'{", ".join(str(j) for j in sorted(_idx.JumpOffYear.unique()))})')

## KS dominance test

In [ ]:
# One-sided KS test for stochastic dominance, as in Bohk-Ewald et al. (2018).
# KS(A, B) = max over t of F_A(t) - F_B(t) on the CF APEs. A dominates B if A's
# APE distribution lies significantly to the left of B's. Each pair is tested in
# both directions. If both are significant the CDFs cross, and the pair counts
# for neither method. A method's KS score is the number of methods it dominates.
#
# P-values come from a country-level paired permutation test, the same one used
# for Table 1 in eval_figures_cohort.ipynb. scipy's KS p-value assumes every APE
# is independent, but errors from the same country are correlated, so it is too
# small. Under H0 (equal accuracy) it is arbitrary which of a country's two APEs
# belongs to which method. Each permutation therefore swaps the two methods'
# APEs for a random subset of countries (each country with probability 1/2) and
# recomputes the KS statistic. The p-value is
# (1 + #{D_perm >= D_obs}) / (1 + N_PERM).
DOMINANCE_ALPHA = 0.05
N_PERM = 10_000
PERM_SEED = 0

names = list(ape_by_model)

# country of each row of ape_by_model (rows are in common_cohorts order)
_cluster = pd.factorize(_idx['Country'])[0]
_n_clusters = _cluster.max() + 1
_flips = np.random.default_rng(PERM_SEED).integers(
    0, 2, size=(N_PERM, _n_clusters)).astype(np.int8)
_PERM_BATCH = 500


def perm_ks(x, y):
    """One-sided KS statistics and their country-permutation p-values.

    Returns (D(x,y), D(y,x), p(x,y), p(y,x)) where D(x,y) = max_t F_x(t) - F_y(t).
    Swapping values between x and y does not change the pooled sample, so it is
    sorted once. F_x - F_y along it is then a running sum of +1 (value is in x)
    and -1 (value is in y), read at the last position of each group of ties.
    """
    n = len(x)
    pooled = np.concatenate([x, y])
    order = np.argsort(pooled, kind='stable')
    vals = pooled[order]
    last_of_tie = np.r_[vals[1:] != vals[:-1], True]
    base = np.where(order < n, 1, -1).astype(np.int32)
    clus = _cluster[order % n]

    obs = np.cumsum(base)[last_of_tie]
    d_xy, d_yx = obs.max() / n, -obs.min() / n
    hits_xy = hits_yx = 0
    for b0 in range(0, N_PERM, _PERM_BATCH):
        swap = _flips[b0:b0 + _PERM_BATCH][:, clus]        # 1 = this row is swapped
        gap = np.cumsum(base * (1 - 2 * swap.astype(np.int32)), axis=1)[:, last_of_tie]
        hits_xy += int((gap.max(axis=1) / n >= d_xy - 1e-12).sum())
        hits_yx += int((-gap.min(axis=1) / n >= d_yx - 1e-12).sum())
    return d_xy, d_yx, (1 + hits_xy) / (1 + N_PERM), (1 + hits_yx) / (1 + N_PERM)


perm_pval = pd.DataFrame(1.0, index=names, columns=names)
for i, a in enumerate(names):
    for b in names[i + 1:]:
        d_ab, d_ba, p_ab, p_ba = perm_ks(ape_by_model[a].values, ape_by_model[b].values)
        # same statistic as scipy's one-sided KS; only the p-value differs
        assert np.isclose(d_ab, ks_2samp(ape_by_model[a].values, ape_by_model[b].values,
                                         alternative='greater').statistic)
        perm_pval.loc[a, b], perm_pval.loc[b, a] = p_ab, p_ba

significant = perm_pval < DOMINANCE_ALPHA
crossover = significant & significant.T
dominates = significant & ~crossover
ks_score = dominates.sum(axis=1)

print(f'Permutation KS: {N_PERM:,} permutations over {_n_clusters} countries '
      f'(seed {PERM_SEED}).')
# flag decisions within 2 Monte Carlo SE of alpha (could flip with another seed)
_mc_se = np.sqrt(perm_pval * (1 - perm_pval) / N_PERM)
for a in names:
    for b in names:
        if a != b and abs(perm_pval.loc[a, b] - DOMINANCE_ALPHA) < 2 * _mc_se.loc[a, b]:
            print(f'  borderline: {a} over {b}, p = {perm_pval.loc[a, b]:.4f}')

## Table A2

In [ ]:
# Table A2
#   KS            number of methods this one dominates (higher is better)
#   50/80/90/95   APE percentiles: the APE below which that share of forecasts falls
#   Mean          mean APE
#   RMSE          root mean square of the APEs
# Rows are ordered by KS, with ties broken by mean APE. Ranks within each column
# are in parentheses (ties share the better rank). "No. of inversions" counts the
# method pairs each column orders the opposite way to the overall ranking.
TABLE_PCTS = (50, 80, 90, 95)

accuracy_table = pd.DataFrame({
    'KS': ks_score,
    **{str(p): {n: np.percentile(s.values, p) for n, s in ape_by_model.items()}
       for p in TABLE_PCTS},
    'Mean': {n: s.mean() for n, s in ape_by_model.items()},
    'RMSE': {n: np.sqrt(np.mean(s.values ** 2)) for n, s in ape_by_model.items()},
}).sort_values(['KS', 'Mean'], ascending=[False, True])

accuracy_ranks = pd.DataFrame(
    {c: accuracy_table[c].rank(method='min', ascending=(c != 'KS')).astype(int)
     for c in accuracy_table.columns},
    index=accuracy_table.index)


def n_inversions(ranks):
    r = list(ranks)
    return sum(1 for i in range(len(r)) for j in range(i + 1, len(r)) if r[i] > r[j])


inversions = accuracy_ranks.apply(n_inversions)


def fmt(c, v, r):
    return f'{v:.0f} ({r})' if c == 'KS' else f'{v:.2f} ({r})'


accuracy_display = pd.DataFrame(
    {c: [fmt(c, v, r) for v, r in zip(accuracy_table[c], accuracy_ranks[c])]
     for c in accuracy_table.columns},
    index=[f'{i + 1}. {name}' for i, name in enumerate(accuracy_table.index)])
accuracy_display.loc['No. of inversions'] = inversions.astype(str)
accuracy_display

In [ ]:
# LaTeX version of Table A2 (needs \usepackage{booktabs})
cols = list(accuracy_table.columns)            # KS, 50, 80, 90, 95, Mean, RMSE
pct_cols = [str(p) for p in TABLE_PCTS]
first_pct = cols.index(pct_cols[0]) + 2        # +1 for the label column, +1 for 1-based
last_pct = cols.index(pct_cols[-1]) + 2

rows = [f'{rank}. {name} & '
        + ' & '.join(fmt(c, row[c], accuracy_ranks.loc[name, c]) for c in cols) + r' \\'
        for rank, (name, row) in enumerate(accuracy_table.iterrows(), start=1)]

latex_table = '\n'.join([
    r'\begin{tabular}{l' + 'r' * len(cols) + '}',
    r'\toprule',
    (' & ' * (first_pct - 1)
     + rf'\multicolumn{{{len(pct_cols)}}}{{c}}{{Threshold APE, \%}}'
     + ' & ' * (len(cols) + 1 - last_pct) + r'\\'),
    rf'\cmidrule(lr){{{first_pct}-{last_pct}}}',
    'Overall ranking & ' + ' & '.join(cols) + r' \\',
    r'\midrule',
    *rows,
    r'\midrule',
    'No. of inversions & ' + ' & '.join(str(inversions[c]) for c in cols) + r' \\',
    r'\bottomrule',
    r'\end{tabular}',
])
print(latex_table)